# GraphMS-Net — Segmentation Validation Audit

**Purpose:** rapid, no-retraining validation of the frozen GraphMS v3.5.1 Hybrid segmentation system.

This notebook does **not** train, tune, select thresholds, or alter the frozen pipeline. It uses the committed Stage16 per-case table and the already-generated final hybrid masks to produce reviewer-facing qualitative evidence: **FLAIR | expert ground truth | final prediction | TP/FP/FN error map** for representative best, median, and worst development cases.

Claim scope remains **five-fold development cross-validation**; this is not external clinical validation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO = '/content/GraphMS-Net'
if not os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git','clone','-q','https://github.com/sath17-o/GraphMS-Net.git',REPO], check=True)
subprocess.run(['git','-C',REPO,'pull','-q','--ff-only'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','nibabel'], check=True)
print('Repository and dependencies ready.')


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

METRICS = Path(REPO) / 'results/stage16/STAGE16_PER_CASE_METRICS.csv'
df = pd.read_csv(METRICS)
assert len(df) == 93, f'Expected 93 final development cases, found {len(df)}'
print('Cases:', len(df), '| folds:', sorted(df.fold.unique().tolist()))
print('DSC range:', float(df.DSC.min()), 'to', float(df.DSC.max()))


## Representative-case selection

Cases are selected **deterministically from the frozen Stage16 DSC column**. No case is hand-picked by visual appearance. The median case is the row closest to the cohort median DSC.


In [ ]:
best = df.loc[df.DSC.idxmax()]
worst = df.loc[df.DSC.idxmin()]
median_value = float(df.DSC.median())
median = df.loc[(df.DSC - median_value).abs().idxmin()]
selected = pd.DataFrame([best, median, worst], index=['BEST','MEDIAN','WORST'])
display(selected[['case','fold','DSC','IoU','Sensitivity','Specificity','HD95_mm','TP','FP','FN']])


In [ ]:
def load_bool(path):
    return np.asarray(nib.load(str(path)).dataobj) > 0

def robust_flair(path):
    x = np.asarray(nib.load(str(path)).dataobj, dtype=np.float32)
    finite = np.isfinite(x)
    if finite.any():
        lo, hi = np.percentile(x[finite], [1, 99])
        x = np.clip((x-lo)/(hi-lo+1e-8), 0, 1)
    return x

def choose_slice(gt, pred):
    burden = np.sum(gt | pred, axis=(0,1))
    return int(np.argmax(burden))

def show_case(row, label):
    case = row['case']
    gt_path = Path(row['gt_path'])
    pred_path = Path(row['mask_path'])
    flair_path = gt_path.parent.parent / 'imagesTr' / f'{case}_0000.nii.gz'
    for p in [gt_path, pred_path, flair_path]:
        assert p.exists(), f'Missing required file: {p}'

    gt = load_bool(gt_path)
    pred = load_bool(pred_path)
    flair = robust_flair(flair_path)
    assert gt.shape == pred.shape == flair.shape, (gt.shape, pred.shape, flair.shape)
    z = choose_slice(gt, pred)

    tp = gt & pred
    fp = (~gt) & pred
    fn = gt & (~pred)
    err = np.zeros(gt.shape, dtype=np.uint8)
    err[tp] = 1; err[fp] = 2; err[fn] = 3

    fig, ax = plt.subplots(1,4,figsize=(16,4), constrained_layout=True)
    for a in ax:
        a.imshow(flair[:,:,z].T, cmap='gray', origin='lower')
        a.axis('off')
    ax[0].set_title(f'{label}: {case}\nFLAIR — slice {z}')
    ax[1].imshow(np.ma.masked_where(~gt[:,:,z].T, gt[:,:,z].T), cmap='autumn', alpha=.65, origin='lower')
    ax[1].set_title('Expert ground truth')
    ax[2].imshow(np.ma.masked_where(~pred[:,:,z].T, pred[:,:,z].T), cmap='winter', alpha=.65, origin='lower')
    ax[2].set_title('Frozen final prediction')
    cmap = ListedColormap(['black','lime','red','deepskyblue'])
    ax[3].imshow(np.ma.masked_where(err[:,:,z].T==0, err[:,:,z].T), cmap=cmap, vmin=0, vmax=3, alpha=.8, origin='lower')
    ax[3].set_title('Error map: TP green | FP red | FN blue')
    fig.suptitle(f"DSC {row['DSC']:.4f} | IoU {row['IoU']:.4f} | Sens {row['Sensitivity']:.4f} | Spec {row['Specificity']:.6f} | HD95 {row['HD95_mm']:.2f} mm", fontsize=11)
    out = Path('/content') / f"SEGMENTATION_{label}_{case}.png"
    fig.savefig(out, dpi=220, bbox_inches='tight')
    plt.show()
    return out

outputs=[]
for label, row in selected.iterrows():
    outputs.append(show_case(row, label))
print('Saved:', *outputs, sep='\n - ')


## Interpretation

The three panels expose both strengths and failure modes without changing the frozen system. **TP** shows correctly segmented lesion voxels, **FP** shows predicted lesion voxels absent from the expert mask, and **FN** shows expert lesion voxels missed by the model.

The final system remains: ResEncM-250 → graph construction → TrueGAT → CNN/GNN hybrid fusion → decoder/head → cross-fitted Stage11. Fold-specific Stage11 thresholds remain frozen (0.40 for folds 0/1/3/4; 0.45 for fold 2), with 26-connectivity and a 10-voxel minimum component size. No morphology is introduced by this audit.


## Cohort-level segmentation diagnostics

This section uses only the **committed 93-case Stage16 table**. It performs no inference, training, tuning, threshold selection, or model modification. Ground-truth lesion burden is used **only for retrospective error analysis**, never as an inference input.

Two summaries are deliberately kept separate:

- **Official Stage16 result:** equal-weight mean of the five outer-fold case means, with sample SD across folds.
- **Case-level descriptive result:** ordinary statistics across all 93 cases, used only to understand the distribution and failure modes.


In [ ]:
audit = df.copy()

audit['GT_lesion_voxels'] = audit['TP'] + audit['FN']
audit['Pred_lesion_voxels'] = audit['TP'] + audit['FP']
audit['Precision'] = audit['TP'] / (audit['TP'] + audit['FP']).replace(0, np.nan)
audit['FNR'] = audit['FN'] / (audit['TP'] + audit['FN']).replace(0, np.nan)
audit['Dominant_error'] = np.where(audit['FN'] > audit['FP'], 'FN-dominant', 'FP-dominant')

metric_cols = ['DSC','IoU','Sensitivity','Specificity','HD95_mm']
fold_case_means = audit.groupby('fold', sort=True)[metric_cols].mean()
official_mean = fold_case_means.mean()
official_sd = fold_case_means.std(ddof=1)

summary = pd.DataFrame({
    'value': {
        'cases': len(audit),
        'official_equal_fold_DSC_mean': official_mean['DSC'],
        'official_equal_fold_DSC_sd': official_sd['DSC'],
        'case_level_DSC_mean': audit['DSC'].mean(),
        'case_level_DSC_median': audit['DSC'].median(),
        'case_level_DSC_min': audit['DSC'].min(),
        'case_level_DSC_max': audit['DSC'].max(),
        'case_level_HD95_median_mm': audit['HD95_mm'].median(),
        'case_level_sensitivity_median': audit['Sensitivity'].median(),
        'case_level_precision_median': audit['Precision'].median(),
        'cases_DSC_lt_0_50': int((audit['DSC'] < 0.50).sum()),
        'cases_DSC_lt_0_60': int((audit['DSC'] < 0.60).sum()),
        'cases_DSC_ge_0_80': int((audit['DSC'] >= 0.80).sum()),
        'cases_HD95_gt_20mm': int((audit['HD95_mm'] > 20).sum()),
        'cases_sensitivity_lt_0_50': int((audit['Sensitivity'] < 0.50).sum()),
    }
})

print('OFFICIAL STAGE16 — equal-weight mean of five outer-fold case means')
display(pd.DataFrame({'mean': official_mean, 'sd_across_folds': official_sd}))
print('CASE-LEVEL DESCRIPTIVE DIAGNOSTICS — 93 development cases')
display(summary)

summary.to_csv('/content/SEGMENTATION_COHORT_SUMMARY.csv')
fold_case_means.to_csv('/content/SEGMENTATION_FOLD_CASE_MEANS.csv')


### Distribution and fold diagnostics

The following plots are descriptive. They are intended to show whether the headline mean is supported consistently across cases and folds, and to make outlier behavior visible.


In [ ]:
# 1) DSC distribution
fig = plt.figure(figsize=(8,5))
plt.hist(audit['DSC'], bins=15)
plt.axvline(float(official_mean['DSC']), linestyle='--', label='Official equal-fold mean')
plt.axvline(float(audit['DSC'].median()), linestyle=':', label='Case median')
plt.xlabel('Dice Similarity Coefficient')
plt.ylabel('Number of cases')
plt.title('GraphMS v3.5.1 — DSC distribution across 93 development cases')
plt.legend()
plt.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_DISTRIBUTION.png', dpi=220, bbox_inches='tight')
plt.show()

# 2) HD95 distribution
fig = plt.figure(figsize=(8,5))
plt.hist(audit['HD95_mm'], bins=15)
plt.axvline(float(audit['HD95_mm'].median()), linestyle='--', label='Case median')
plt.xlabel('HD95 (mm)')
plt.ylabel('Number of cases')
plt.title('GraphMS v3.5.1 — HD95 distribution across 93 development cases')
plt.legend()
plt.tight_layout()
fig.savefig('/content/SEGMENTATION_HD95_DISTRIBUTION.png', dpi=220, bbox_inches='tight')
plt.show()

# 3) Fold-wise DSC
fig = plt.figure(figsize=(8,5))
audit.boxplot(column='DSC', by='fold', grid=False)
plt.suptitle('')
plt.title('Fold-wise DSC distribution')
plt.xlabel('Outer fold')
plt.ylabel('DSC')
plt.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_BY_FOLD.png', dpi=220, bbox_inches='tight')
plt.show()


### Lesion-burden failure analysis

MS lesion segmentation can be disproportionately difficult when the expert lesion burden is small. This analysis tests that **descriptively** using only the frozen OOF confusion counts. It does not change the model and does not establish causation.


In [ ]:
audit['burden_quartile'] = pd.qcut(
    audit['GT_lesion_voxels'],
    q=4,
    labels=['Q1 smallest','Q2','Q3','Q4 largest']
)

burden_summary = (
    audit.groupby('burden_quartile', observed=True)
         .agg(
             n=('case','size'),
             GT_vox_min=('GT_lesion_voxels','min'),
             GT_vox_max=('GT_lesion_voxels','max'),
             DSC_mean=('DSC','mean'),
             DSC_median=('DSC','median'),
             Sensitivity_mean=('Sensitivity','mean'),
             HD95_median_mm=('HD95_mm','median'),
         )
)

r_logburden_dsc = float(np.corrcoef(np.log10(audit['GT_lesion_voxels'] + 1), audit['DSC'])[0,1])
print('Pearson r(log10 GT lesion voxels, DSC) =', round(r_logburden_dsc, 4))
display(burden_summary)
burden_summary.to_csv('/content/SEGMENTATION_BURDEN_QUARTILES.csv')

fig = plt.figure(figsize=(8,5))
plt.scatter(audit['GT_lesion_voxels'], audit['DSC'], alpha=0.75)
plt.xscale('log')
x = np.log10(audit['GT_lesion_voxels'].to_numpy() + 1)
y = audit['DSC'].to_numpy()
coef = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 200)
plt.plot(10**xs - 1, np.polyval(coef, xs), linestyle='--')
plt.xlabel('Expert lesion burden (voxels, log scale)')
plt.ylabel('DSC')
plt.title('DSC versus expert lesion burden — descriptive development analysis')
plt.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_VS_LESION_BURDEN.png', dpi=220, bbox_inches='tight')
plt.show()

fig = plt.figure(figsize=(8,5))
burden_summary['DSC_mean'].plot(kind='bar')
plt.ylabel('Mean DSC')
plt.xlabel('Ground-truth lesion-burden quartile')
plt.title('Mean DSC by expert lesion-burden quartile')
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig('/content/SEGMENTATION_DSC_BY_BURDEN_QUARTILE.png', dpi=220, bbox_inches='tight')
plt.show()


### Worst-case table and dominant error direction

The table below exposes the lowest-DSC cases rather than hiding them. `FN-dominant` means missed lesion voxels exceed false-positive lesion voxels for that case; `FP-dominant` means the reverse.


In [ ]:
worst10 = (
    audit.sort_values('DSC', ascending=True)
         .head(10)
         [['case','fold','DSC','IoU','Sensitivity','Precision','HD95_mm',
           'GT_lesion_voxels','TP','FP','FN','Dominant_error']]
         .reset_index(drop=True)
)

display(worst10)
worst10.to_csv('/content/SEGMENTATION_WORST10.csv', index=False)

low_dsc = audit[audit['DSC'] < 0.60]
print('Cases with DSC < 0.60:', len(low_dsc))
print('FN-dominant among DSC < 0.60:', int((low_dsc['Dominant_error'] == 'FN-dominant').sum()))
print('FP-dominant among DSC < 0.60:', int((low_dsc['Dominant_error'] == 'FP-dominant').sum()))

print('\nFrozen-table diagnostic summary')
print(f"- Official equal-fold DSC: {official_mean['DSC']:.6f} ± {official_sd['DSC']:.6f}")
print(f"- Case-level median DSC: {audit['DSC'].median():.6f}")
print(f"- DSC < 0.50: {(audit['DSC'] < 0.50).sum()} / {len(audit)} cases")
print(f"- HD95 > 20 mm: {(audit['HD95_mm'] > 20).sum()} / {len(audit)} cases")
print(f"- Smallest-burden quartile mean DSC: {burden_summary.iloc[0]['DSC_mean']:.6f}")
print(f"- Largest-burden quartile mean DSC: {burden_summary.iloc[-1]['DSC_mean']:.6f}")
print(f"- r(log lesion burden, DSC): {r_logburden_dsc:.4f}")


## What the diagnostic can establish

This audit can establish **where** the frozen development system fails; it does not retroactively change the model. In particular, if the lowest-DSC cases cluster at low expert lesion burden and are FN-dominant, the defensible interpretation is that **small/subtle lesion sensitivity is a principal residual weakness of the frozen segmentation system**. That is a development-set failure-mode observation, not a clinical-generalization claim.

The correct submission response is therefore to show this limitation transparently alongside the strong median/high-performing cases and the leakage-controlled five-fold aggregate—not to modify the frozen preprocessing or tune Stage11 again on the same outer-fold evidence.


In [ ]:
# Export a compact reviewer-facing diagnostics bundle.
from zipfile import ZipFile, ZIP_DEFLATED

artifacts = [
    '/content/SEGMENTATION_COHORT_SUMMARY.csv',
    '/content/SEGMENTATION_FOLD_CASE_MEANS.csv',
    '/content/SEGMENTATION_BURDEN_QUARTILES.csv',
    '/content/SEGMENTATION_WORST10.csv',
    '/content/SEGMENTATION_DSC_DISTRIBUTION.png',
    '/content/SEGMENTATION_HD95_DISTRIBUTION.png',
    '/content/SEGMENTATION_DSC_BY_FOLD.png',
    '/content/SEGMENTATION_DSC_VS_LESION_BURDEN.png',
    '/content/SEGMENTATION_DSC_BY_BURDEN_QUARTILE.png',
]

for label, row in selected.iterrows():
    artifacts.append(f"/content/SEGMENTATION_{label}_{row['case']}.png")

artifacts = [Path(x) for x in artifacts if Path(x).exists()]
zip_path = Path('/content/GraphMS_Segmentation_Diagnostics.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as zf:
    for a in artifacts:
        zf.write(a, arcname=a.name)

print('Diagnostics bundle:', zip_path)
print('Included files:', len(artifacts))
